# IDHM (Atlas / Base dos Dados)

Prepara o Índice de Desenvolvimento Humano Municipal usado pelo livro de IVS/IDH.
As definições das variáveis do livro publicado estão em **dictionary.csv**.


## Importações


In [4]:
import re
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Optional
from unidecode import unidecode


## Configuração Inicial

In [5]:
def resolve_repo_root() -> Path:
    """Localiza a raiz do projeto que contém o diretório local de dados."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"

RAW_IDHM_DIR = BASE_DIR / "raw" / "idhm"
TRUSTED_IDHM_DIR = BASE_DIR / "trusted" / "idhm"
TRUSTED_IBGE_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_IDHM_DIR, TRUSTED_IDHM_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Leitura de Arquivos

In [6]:
path_municipios_bd = RAW_IDHM_DIR / "br_bd_diretorios_brasil_municipio.csv.gz"
path_idhm_bd = RAW_IDHM_DIR / "mundo_onu_adh_municipio.csv.gz"

df_municipios_bd = pd.read_csv(path_municipios_bd, dtype="string", compression="gzip")
df_idhm_raw = pd.read_csv(path_idhm_bd, dtype="string", compression="gzip")


In [7]:
print(df_municipios_bd.shape)
print(df_idhm_raw.shape)

display(df_municipios_bd.head())
display(df_idhm_raw.head())


(5571, 27)
(16695, 230)


,id_municipio,id_municipio_6,id_municipio_tse,id_municipio_rf,id_municipio_bcb,nome,capital_uf,id_comarca,id_regiao_saude,nome_regiao_saude,...,nome_mesorregiao,id_regiao_metropolitana,nome_regiao_metropolitana,ddd,id_uf,sigla_uf,nome_uf,nome_regiao,amazonia_legal,centroide
0,5101837,510183,<NA>,<NA>,<NA>,Boa Esperança do Norte,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,51,MT,Mato Grosso,Centro-Oeste,1,POINT(-54.9417429754238 -13.463197935201)
1,1100809,110080,477,681,46961,Candeias do Jamari,0,1100205,11004,Madeira-Mamoré,...,Madeira-Guaporé,101,Região Metropolitana de Porto Velho,69,11,RO,Rondônia,Norte,1,POINT(-63.3254198532114 -8.88702392955617)
2,1100338,110033,434,47,44516,Nova Mamoré,0,1100106,11004,Madeira-Mamoré,...,Madeira-Guaporé,<NA>,<NA>,69,11,RO,Rondônia,Norte,1,POINT(-64.6295025002404 -10.3822863210755)
3,1100205,110020,35,3,30719,Porto Velho,1,1100205,11004,Madeira-Mamoré,...,Madeira-Guaporé,101,Região Metropolitana de Porto Velho,69,11,RO,Rondônia,Norte,1,POINT(-64.3041357999869 -9.15394033687075)
4,1101104,110110,493,683,46851,Itapuã do Oeste,0,1100205,11004,Madeira-Mamoré,...,Madeira-Guaporé,<NA>,<NA>,69,11,RO,Rondônia,Norte,1,POINT(-63.044960463046 -9.17019402601564)


,ano,id_municipio,expectativa_vida,fecundidade_total,mortalidade_1,mortalidade_5,razao_dependencia,prob_sobrevivencia_40,prob_sobrevivencia_60,taxa_envelhecimento,...,pia,pia_10_14,pia_15_17,pia_18_mais,indice_escolaridade,indice_frequencia_escolar,idhm,idhm_e,idhm_l,idhm_r
0,1991,1100015,62.01,4.08,45.58,58.05,73.5,83.81,66.87,1.82,...,<NA>,<NA>,<NA>,<NA>,0.117,0.109,0.329,0.112,0.617,0.516
1,1991,1100023,66.02,3.72,32.39,41.41,69.97,88.08,74.23,1.82,...,<NA>,<NA>,<NA>,<NA>,0.2,0.199,0.432,0.199,0.684,0.593
2,1991,1100031,63.16,3.89,41.52,52.94,77.23,85.09,69,2.59,...,<NA>,<NA>,<NA>,<NA>,0.111,0.106,0.309,0.108,0.636,0.43
3,1991,1100049,65.03,3.81,35.37,45.19,71.11,87.08,72.44,2.47,...,<NA>,<NA>,<NA>,<NA>,0.176,0.169,0.407,0.171,0.667,0.593
4,1991,1100056,62.73,3.55,43,54.82,71.09,84.62,68.2,2.46,...,<NA>,<NA>,<NA>,<NA>,0.152,0.175,0.386,0.167,0.629,0.547


In [8]:
df_municipios_bd.columns.tolist()


['id_municipio',
 'id_municipio_6',
 'id_municipio_tse',
 'id_municipio_rf',
 'id_municipio_bcb',
 'nome',
 'capital_uf',
 'id_comarca',
 'id_regiao_saude',
 'nome_regiao_saude',
 'id_regiao_imediata',
 'nome_regiao_imediata',
 'id_regiao_intermediaria',
 'nome_regiao_intermediaria',
 'id_microrregiao',
 'nome_microrregiao',
 'id_mesorregiao',
 'nome_mesorregiao',
 'id_regiao_metropolitana',
 'nome_regiao_metropolitana',
 'ddd',
 'id_uf',
 'sigla_uf',
 'nome_uf',
 'nome_regiao',
 'amazonia_legal',
 'centroide']

In [9]:
df_idhm_raw.columns.tolist()


['ano',
 'id_municipio',
 'expectativa_vida',
 'fecundidade_total',
 'mortalidade_1',
 'mortalidade_5',
 'razao_dependencia',
 'prob_sobrevivencia_40',
 'prob_sobrevivencia_60',
 'taxa_envelhecimento',
 'expectativa_anos_estudo',
 'taxa_analfabetismo_11_a_14',
 'taxa_analfabetismo_15_a_17',
 'taxa_analfabetismo_15_mais',
 'taxa_analfabetismo_18_a_24',
 'taxa_analfabetismo_18_mais',
 'taxa_analfabetismo_25_a_29',
 'taxa_analfabetismo_25_mais',
 'taxa_atraso_0_basico',
 'taxa_atraso_0_fundamental',
 'taxa_atraso_0_medio',
 'taxa_atraso_1_basico',
 'taxa_atraso_1_fundamental',
 'taxa_atraso_1_medio',
 'taxa_atraso_2_basico',
 'taxa_atraso_2_fundamental',
 'taxa_atraso_2_medio',
 'taxa_freq_bruta_basico',
 'taxa_freq_bruta_fundamental',
 'taxa_freq_bruta_medio',
 'taxa_freq_bruta_pre',
 'taxa_freq_bruta_superior',
 'taxa_freq_liquida_basico',
 'taxa_freq_liquida_fundamental',
 'taxa_freq_liquida_medio',
 'taxa_freq_liquida_pre',
 'taxa_freq_liquida_superior',
 'taxa_freq_0_3',
 'taxa_freq_

## Normalizar nomes das colunas

In [10]:
def normalizar_nome_coluna(col: str) -> str:
    col = str(col).strip()
    col = unidecode(col)
    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalizar_nome_coluna(col) for col in df.columns]
    return df


In [11]:
df_municipios_bd = normalizar_colunas(df_municipios_bd)
df_idhm_raw = normalizar_colunas(df_idhm_raw)

print(df_municipios_bd.columns.tolist())
print(df_idhm_raw.columns.tolist())


['id_municipio', 'id_municipio_6', 'id_municipio_tse', 'id_municipio_rf', 'id_municipio_bcb', 'nome', 'capital_uf', 'id_comarca', 'id_regiao_saude', 'nome_regiao_saude', 'id_regiao_imediata', 'nome_regiao_imediata', 'id_regiao_intermediaria', 'nome_regiao_intermediaria', 'id_microrregiao', 'nome_microrregiao', 'id_mesorregiao', 'nome_mesorregiao', 'id_regiao_metropolitana', 'nome_regiao_metropolitana', 'ddd', 'id_uf', 'sigla_uf', 'nome_uf', 'nome_regiao', 'amazonia_legal', 'centroide']
['ano', 'id_municipio', 'expectativa_vida', 'fecundidade_total', 'mortalidade_1', 'mortalidade_5', 'razao_dependencia', 'prob_sobrevivencia_40', 'prob_sobrevivencia_60', 'taxa_envelhecimento', 'expectativa_anos_estudo', 'taxa_analfabetismo_11_a_14', 'taxa_analfabetismo_15_a_17', 'taxa_analfabetismo_15_mais', 'taxa_analfabetismo_18_a_24', 'taxa_analfabetismo_18_mais', 'taxa_analfabetismo_25_a_29', 'taxa_analfabetismo_25_mais', 'taxa_atraso_0_basico', 'taxa_atraso_0_fundamental', 'taxa_atraso_0_medio', 'ta

## Preparar a dimensão de municípios da Base dos Dados

In [12]:
def preparar_dim_municipios_bd(df_municipios_bd: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara dimensão municipal da Base dos Dados.

    Espera uma coluna id_municipio.
    Retorna uma dimensão com codigo_ibge e informações territoriais.
    """
    df = df_municipios_bd.copy()
    df = normalizar_colunas(df)

    if "id_municipio" not in df.columns:
        raise ValueError(f"Coluna id_municipio não encontrada. Colunas disponíveis: {df.columns.tolist()}")

    df["codigo_ibge"] = (
        df["id_municipio"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    # Renomeia colunas comuns da Base dos Dados, se existirem
    rename_map = {}

    if "nome" in df.columns:
        rename_map["nome"] = "municipio"

    if "sigla_uf" in df.columns:
        rename_map["sigla_uf"] = "uf"

    if "nome_uf" in df.columns:
        rename_map["nome_uf"] = "nome_uf"

    if "regiao" in df.columns:
        rename_map["regiao"] = "regiao"

    if "nome_regiao_imediata" in df.columns:
        rename_map["nome_regiao_imediata"] = "regiao_imediata"

    if "nome_regiao_intermediaria" in df.columns:
        rename_map["nome_regiao_intermediaria"] = "regiao_intermediaria"

    df = df.rename(columns=rename_map)

    cols_keep = [
        "codigo_ibge",
        "id_municipio",
        "municipio",
        "uf",
        "nome_uf",
        "regiao",
        "regiao_imediata",
        "regiao_intermediaria",
        "id_uf",
        "id_regiao_imediata",
        "id_regiao_intermediaria",
        "id_municipio_6",
    ]

    cols_keep = [col for col in cols_keep if col in df.columns]

    df = (
        df[cols_keep]
        .drop_duplicates("codigo_ibge")
        .sort_values("codigo_ibge")
        .reset_index(drop=True)
    )

    return df


In [13]:
dim_municipios_bd = preparar_dim_municipios_bd(df_municipios_bd)

dim_municipios_bd.head()


,codigo_ibge,id_municipio,municipio,uf,nome_uf,regiao_imediata,regiao_intermediaria,id_uf,id_regiao_imediata,id_regiao_intermediaria,id_municipio_6
0,1100015,1100015,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,11,110005,1102,110001
1,1100023,1100023,Ariquemes,RO,Rondônia,Ariquemes,Porto Velho,11,110002,1101,110002
2,1100031,1100031,Cabixi,RO,Rondônia,Vilhena,Ji-Paraná,11,110006,1102,110003
3,1100049,1100049,Cacoal,RO,Rondônia,Cacoal,Ji-Paraná,11,110005,1102,110004
4,1100056,1100056,Cerejeiras,RO,Rondônia,Vilhena,Ji-Paraná,11,110006,1102,110005


## Salvar

In [14]:
# dim_municipios_bd.to_parquet(
#     TRUSTED_IDHM_DIR / "dim_municipios_base_dos_dados.parquet",
#     index=False
# )

# dim_municipios_bd.to_csv(
#     TRUSTED_IDHM_DIR / "dim_municipios_base_dos_dados.csv",
#     index=False,
#     encoding="utf-8"
# )


## Base IDHM

In [15]:
def selecionar_coluna(
    df: pd.DataFrame,
    candidatos: list[str],
    obrigatoria: bool = True,
    nome_variavel: str = ""
) -> Optional[str]:
    cols = set(df.columns)
    candidatos_norm = [normalizar_nome_coluna(c) for c in candidatos]

    for col in candidatos_norm:
        if col in cols:
            return col

    if obrigatoria:
        raise ValueError(
            f"Não encontrei coluna para {nome_variavel}. "
            f"Candidatos testados: {candidatos_norm}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    return None


def parse_decimal(s: pd.Series) -> pd.Series:
    """
    Interpreta textos decimais com vírgula ou ponto como separador decimal.
    """
    return (
        s.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "-": pd.NA,
            "...": pd.NA,
        })
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )


In [16]:
def padronizar_idhm_base_dos_dados(df_idhm_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza a base mundo_onu_adh_municipio baixada da Base dos Dados.

    Saída:
        codigo_ibge
        ano
        idhm
        idhm_educacao
        idhm_longevidade
        idhm_renda
    """
    df = normalizar_colunas(df_idhm_raw)

    col_id_municipio = selecionar_coluna(
        df,
        ["id_municipio", "codigo_ibge", "cod_ibge"],
        obrigatoria=True,
        nome_variavel="id_municipio"
    )

    col_ano = selecionar_coluna(
        df,
        ["ano"],
        obrigatoria=True,
        nome_variavel="ano"
    )

    col_idhm = selecionar_coluna(
        df,
        ["idhm", "idh_m"],
        obrigatoria=True,
        nome_variavel="idhm"
    )

    col_idhm_educacao = selecionar_coluna(
        df,
        ["idhm_educacao", "idhm_e", "idh_m_educacao"],
        obrigatoria=False,
        nome_variavel="idhm_educacao"
    )

    col_idhm_longevidade = selecionar_coluna(
        df,
        ["idhm_longevidade", "idhm_l", "idh_m_longevidade"],
        obrigatoria=False,
        nome_variavel="idhm_longevidade"
    )

    col_idhm_renda = selecionar_coluna(
        df,
        ["idhm_renda", "idhm_r", "idh_m_renda"],
        obrigatoria=False,
        nome_variavel="idhm_renda"
    )

    out = pd.DataFrame()

    out["codigo_ibge"] = (
        df[col_id_municipio]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out["ano"] = pd.to_numeric(
        df[col_ano],
        errors="coerce"
    ).astype("Int64")

    out["idhm"] = parse_decimal(df[col_idhm])

    out["idhm_educacao"] = (
        parse_decimal(df[col_idhm_educacao])
        if col_idhm_educacao is not None
        else np.nan
    )

    out["idhm_longevidade"] = (
        parse_decimal(df[col_idhm_longevidade])
        if col_idhm_longevidade is not None
        else np.nan
    )

    out["idhm_renda"] = (
        parse_decimal(df[col_idhm_renda])
        if col_idhm_renda is not None
        else np.nan
    )

    out = (
        out
        .dropna(subset=["codigo_ibge", "ano", "idhm"])
        .drop_duplicates(["codigo_ibge", "ano"])
        .sort_values(["codigo_ibge", "ano"])
        .reset_index(drop=True)
    )

    return out


In [17]:
idhm_municipio = padronizar_idhm_base_dos_dados(df_idhm_raw)

idhm_municipio.head()


,codigo_ibge,ano,idhm,idhm_educacao,idhm_longevidade,idhm_renda
0,1100015,1991,0.329,0.112,0.617,0.516
1,1100015,2000,0.483,0.262,0.698,0.617
2,1100015,2010,0.641,0.526,0.763,0.657
3,1100023,1991,0.432,0.199,0.684,0.593
4,1100023,2000,0.556,0.343,0.742,0.674


In [18]:
(
    idhm_municipio
    .groupby("ano")
    .agg(
        qtd_municipios=("codigo_ibge", "nunique"),
        idhm_medio=("idhm", "mean"),
        idhm_min=("idhm", "min"),
        idhm_max=("idhm", "max"),
        idhm_educacao_medio=("idhm_educacao", "mean"),
        idhm_longevidade_medio=("idhm_longevidade", "mean"),
        idhm_renda_medio=("idhm_renda", "mean"),
    )
    .reset_index()
    .sort_values("ano")
)


,ano,qtd_municipios,idhm_medio,idhm_min,idhm_max,idhm_educacao_medio,idhm_longevidade_medio,idhm_renda_medio
0,1991,5565,0.381372,0.12,0.697,0.178705,0.645658,0.514515
1,2000,5565,0.523481,0.208,0.82,0.354197,0.723611,0.575718
2,2010,5565,0.659157,0.418,0.862,0.559094,0.801564,0.642873


## IDHM e Base dos Dados

In [19]:
idhm_municipio_bd = (
    idhm_municipio
    .merge(
        dim_municipios_bd,
        on="codigo_ibge",
        how="left"
    )
)

idhm_municipio_bd.head()


,codigo_ibge,ano,idhm,idhm_educacao,idhm_longevidade,idhm_renda,id_municipio,municipio,uf,nome_uf,regiao_imediata,regiao_intermediaria,id_uf,id_regiao_imediata,id_regiao_intermediaria,id_municipio_6
0,1100015,1991,0.329,0.112,0.617,0.516,1100015,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,11,110005,1102,110001
1,1100015,2000,0.483,0.262,0.698,0.617,1100015,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,11,110005,1102,110001
2,1100015,2010,0.641,0.526,0.763,0.657,1100015,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,11,110005,1102,110001
3,1100023,1991,0.432,0.199,0.684,0.593,1100023,Ariquemes,RO,Rondônia,Ariquemes,Porto Velho,11,110002,1101,110002
4,1100023,2000,0.556,0.343,0.742,0.674,1100023,Ariquemes,RO,Rondônia,Ariquemes,Porto Velho,11,110002,1101,110002


In [20]:
cols_ordem = [
    "codigo_ibge",
    "id_municipio",
    "ano",
    "municipio",
    "uf",
    "nome_uf",
    "regiao",
    "regiao_imediata",
    "regiao_intermediaria",
    "idhm",
    "idhm_educacao",
    "idhm_longevidade",
    "idhm_renda",
]

cols_ordem = [col for col in cols_ordem if col in idhm_municipio_bd.columns]

idhm_municipio_bd = idhm_municipio_bd[
    cols_ordem + [col for col in idhm_municipio_bd.columns if col not in cols_ordem]
]

idhm_municipio_bd.head()


,codigo_ibge,id_municipio,ano,municipio,uf,nome_uf,regiao_imediata,regiao_intermediaria,idhm,idhm_educacao,idhm_longevidade,idhm_renda,id_uf,id_regiao_imediata,id_regiao_intermediaria,id_municipio_6
0,1100015,1100015,1991,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,0.329,0.112,0.617,0.516,11,110005,1102,110001
1,1100015,1100015,2000,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,0.483,0.262,0.698,0.617,11,110005,1102,110001
2,1100015,1100015,2010,Alta Floresta D'Oeste,RO,Rondônia,Cacoal,Ji-Paraná,0.641,0.526,0.763,0.657,11,110005,1102,110001
3,1100023,1100023,1991,Ariquemes,RO,Rondônia,Ariquemes,Porto Velho,0.432,0.199,0.684,0.593,11,110002,1101,110002
4,1100023,1100023,2000,Ariquemes,RO,Rondônia,Ariquemes,Porto Velho,0.556,0.343,0.742,0.674,11,110002,1101,110002


In [21]:
# idhm_municipio.to_parquet(
#     TRUSTED_IDHM_DIR / "idhm_municipio_tratado.parquet",
#     index=False
# )

# idhm_municipio_bd.to_parquet(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_diretorio_bd.parquet",
#     index=False
# )

# idhm_municipio_bd.to_csv(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_diretorio_bd.csv",
#     index=False,
#     encoding="utf-8"
# )


## Cruzar com dim_municipio do IBGE
Há 4 ou 5 referências nos municípios e 3 referências de IDHM. Precisa corrigir.

In [22]:
def carregar_dim_municipio_ibge(
    path_dim: Path = TRUSTED_IBGE_DIR / "dim_municipio.parquet"
) -> pd.DataFrame:
    """
    Carrega a dimensão oficial de municípios do IBGE usada no projeto.
    """
    dim = pd.read_parquet(path_dim).copy()

    dim["codigo_ibge"] = (
        dim["codigo_ibge"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    cols = [
        "codigo_ibge",
        "municipio",
        "uf",
        "nome_uf",
        "regiao",
        "sigla_regiao",
        "regiao_imediata",
        "regiao_intermediaria",
        "codigo_uf",
        "codigo_regiao",
        "codigo_regiao_imediata",
        "codigo_regiao_intermediaria",
    ]

    cols = [col for col in cols if col in dim.columns]

    return dim[cols].drop_duplicates("codigo_ibge")


In [23]:
dim_municipio_ibge = carregar_dim_municipio_ibge(
    TRUSTED_IBGE_DIR / "dim_municipio.parquet"
)

idhm_municipio_ibge = (
    idhm_municipio
    .merge(
        dim_municipio_ibge,
        on="codigo_ibge",
        how="left"
    )
)

idhm_municipio_ibge.head()


,codigo_ibge,ano,idhm,idhm_educacao,idhm_longevidade,idhm_renda,municipio,uf,nome_uf,regiao,sigla_regiao,regiao_imediata,regiao_intermediaria,codigo_uf,codigo_regiao,codigo_regiao_imediata,codigo_regiao_intermediaria
0,1100015,1991,0.329,0.112,0.617,0.516,Alta Floresta D'Oeste,RO,Rondônia,Norte,N,Cacoal,Ji-Paraná,11.0,1.0,110005,1102
1,1100015,2000,0.483,0.262,0.698,0.617,Alta Floresta D'Oeste,RO,Rondônia,Norte,N,Cacoal,Ji-Paraná,11.0,1.0,110005,1102
2,1100015,2010,0.641,0.526,0.763,0.657,Alta Floresta D'Oeste,RO,Rondônia,Norte,N,Cacoal,Ji-Paraná,11.0,1.0,110005,1102
3,1100023,1991,0.432,0.199,0.684,0.593,Ariquemes,RO,Rondônia,Norte,N,Ariquemes,Porto Velho,11.0,1.0,110002,1101
4,1100023,2000,0.556,0.343,0.742,0.674,Ariquemes,RO,Rondônia,Norte,N,Ariquemes,Porto Velho,11.0,1.0,110002,1101


In [24]:
# idhm_municipio_ibge.to_parquet(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_dim_ibge.parquet",
#     index=False
# )

# idhm_municipio_ibge.to_csv(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_dim_ibge.csv",
#     index=False,
#     encoding="utf-8"
# )
